<a href="https://colab.research.google.com/github/vipinkumarcp/NLP_Practice/blob/main/fakenewsBidirectionalLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
import pandas as pd

In [45]:
fake_news = pd.read_csv('/content/Fake.csv')

In [46]:
true_news = pd.read_csv('/content/True.csv')

In [47]:
fake_news.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [48]:
true_news.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [49]:
fake_news['label']=1

In [50]:
true_news['label']=0

In [51]:

df = pd.concat([true_news,fake_news],ignore_index=True)

df = df.sample(frac=1).reset_index(drop=True)


In [52]:
df.head()

,title,text,subject,date,label
0,IF THESE CELEBRITIES ARE “With Her” Then Why I...,Hillary gives pay-to-play a whole new meaning ...,left-news,"Oct 30, 2016",1
1,Children Will Suffer: Trumpcare Would Cut $46...,"Children, the most vulnerable of Americans, wi...",News,"May 18, 2017",1
2,BRAVO! WATCH TED CRUZ SLAM The Lies Of Leftist...,Watch Ted Cruz completely annihilate the Lefti...,politics,"Jan 10, 2017",1
3,Republican Trump says 'system is rigged' after...,WASHINGTON (Reuters) - U.S. Republican preside...,politicsNews,"July 5, 2016",0
4,"Unlike Hillary, Cruz’s Campaign May Have ACTU...",In Ted Cruz s last-ditch effort to scare prima...,News,"February 1, 2016",1


In [53]:
df.isnull().sum()

,0
title,0
text,0
subject,0
date,0
label,0


In [54]:
from re import X
#indepent features
X = df.drop('label',axis=1)

In [55]:
y = df['label']

In [56]:
y.value_counts()

,count
label,
1,23481
0,21417


In [57]:
import tensorflow as tf

In [58]:
tf.__version__

'2.19.0'

In [59]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM,Bidirectional
from tensorflow.keras.layers import Dense,Input

In [60]:
## Vocabulary size

voc_size = 5000

In [61]:
messages=X.copy()

In [62]:
messages

,title,text,subject,date
0,IF THESE CELEBRITIES ARE “With Her” Then Why I...,Hillary gives pay-to-play a whole new meaning ...,left-news,"Oct 30, 2016"
1,Children Will Suffer: Trumpcare Would Cut $46...,"Children, the most vulnerable of Americans, wi...",News,"May 18, 2017"
2,BRAVO! WATCH TED CRUZ SLAM The Lies Of Leftist...,Watch Ted Cruz completely annihilate the Lefti...,politics,"Jan 10, 2017"
3,Republican Trump says 'system is rigged' after...,WASHINGTON (Reuters) - U.S. Republican preside...,politicsNews,"July 5, 2016"
4,"Unlike Hillary, Cruz’s Campaign May Have ACTU...",In Ted Cruz s last-ditch effort to scare prima...,News,"February 1, 2016"
...,...,...,...,...
44893,Do not expect postcard-sized tax return from R...,"WASHINGTON (Reuters) - Since mid-2016, U.S. Ho...",politicsNews,"December 14, 2017"
44894,Bill Maher Explains Socialism To Republicans ...,"Until recently, socialism has been one of th...",News,"June 4, 2016"
44895,Convicted 'bookkeeper of Auschwitz' challenges...,BERLIN (Reuters) - A 96-year-old German convic...,worldnews,"December 19, 2017"
44896,Senators close to bipartisan deal on health ex...,WASHINGTON (Reuters) - Two U.S. senators from ...,politicsNews,"September 28, 2017"


In [63]:
import nltk
import re
from nltk.corpus import stopwords

In [64]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [65]:

### Dataset Preprocessing
from nltk.stem.porter import PorterStemmer ##stemming purpose
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review = review.lower()
    review = review.split()

    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)


In [66]:
corpus[:5]

['celebr hillari pay big buck perform fundrais',
 'children suffer trumpcar would cut b healthcar fund kid',
 'bravo watch ted cruz slam lie leftist attack senat jeff session video',
 'republican trump say system rig clinton email announc',
 'unlik hillari cruz campaign may actual broken law iowa']

In [67]:
onehot_repr =[one_hot(words,voc_size) for words in corpus]

In [68]:
print(onehot_repr[:4])

[[426, 2487, 4132, 4898, 1002, 2758, 1828], [3477, 1569, 35, 452, 427, 303, 3524, 3152, 4120], [2274, 715, 1889, 4676, 2537, 4226, 2958, 3710, 3826, 2301, 2213, 1853], [1202, 1034, 4199, 2078, 2240, 543, 818, 3453]]


In [69]:
#embedding represenation

sent_length =20
embedded_docs = pad_sequences(onehot_repr,padding='pre',maxlen=sent_length)


In [71]:
embedded_docs[:2]

array([[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,  426, 2487, 4132, 4898, 1002, 2758, 1828],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
        3477, 1569,   35,  452,  427,  303, 3524, 3152, 4120]],
      dtype=int32)

In [72]:


## Creating model
embedding_vector_features=40 ##features representation
model=Sequential()
model.add(Input(shape=(sent_length,)))
model.add(Embedding(voc_size,embedding_vector_features))
model.add(Bidirectional(LSTM(100)))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 40)         │       200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 200)            │       112,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           201 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 313,001 (1.19 MB)

 Trainable params: 313,001 (1.19 MB)

 Non-trainable params: 0 (0.00 B)

None


In [73]:

len(embedded_docs),y.shape

(44898, (44898,))

In [74]:

import numpy as np
X_final = np.array(embedded_docs)
y_final = np.array(y)

In [75]:


X_final.shape

(44898, 20)

In [76]:
y_final.shape

(44898,)

In [77]:


from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33)

In [78]:

### Finally Training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9050 - loss: 0.2267 - val_accuracy: 0.9336 - val_loss: 0.1699
Epoch 2/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9461 - loss: 0.1376 - val_accuracy: 0.9404 - val_loss: 0.1519
Epoch 3/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9606 - loss: 0.1047 - val_accuracy: 0.9384 - val_loss: 0.1722
Epoch 4/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9708 - loss: 0.0817 - val_accuracy: 0.9384 - val_loss: 0.1678
Epoch 5/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9761 - loss: 0.0669 - val_accuracy: 0.9387 - val_loss: 0.1841
Epoch 6/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9820 - loss: 0.0537 - val_accuracy: 0.9391 - val_loss: 0.2121
Epoch 7/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - accuracy: 0.9866 - loss: 0.0416 - val_accuracy: 0.9316 - val_loss: 0.2505
Epoch 8/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9883 - loss: 0.0368 - val_accura

In [79]:
y_pred=model.predict(X_test)


464/464 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step


In [80]:
y_pred

array([[9.9998522e-01],
       [9.9999523e-01],
       [2.6974292e-05],
       ...,
       [4.6330453e-05],
       [1.0000000e+00],
       [9.9999881e-01]], dtype=float32)

In [81]:

y_pred=np.where(y_pred > 0.6, 1,0) ##AUC ROC Curve

In [82]:

from sklearn.metrics import confusion_matrix

In [83]:


confusion_matrix(y_test,y_pred)


array([[6662,  374],
       [ 585, 7196]])

In [84]:

from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)


0.9352770466356213

In [85]:

from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.92      0.95      0.93      7036
           1       0.95      0.92      0.94      7781

    accuracy                           0.94     14817
   macro avg       0.93      0.94      0.94     14817
weighted avg       0.94      0.94      0.94     14817

